# Exploratory Data Analysis

The goal of this notebook is to understand the dataset well enough to make defensible
preprocessing decisions for the training pipeline. Each section ends with a decision
that propagates to `churn_prediction/preprocessing.py`.

A summary of all decisions sits at the bottom of the notebook.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df = pd.read_csv("../data/training/telco-churn-dataset.csv")
df.head()

## Shape, dtypes, and missing values

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

No nulls reported by pandas, but `TotalCharges` has dtype `object` despite
looking numeric. This is a hint that some rows contain non-numeric values (in this
dataset, blank strings). Coercing to numeric will reveal them.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [ ]:
df.isnull().sum()

11 rows surface as null after coercion, these were stored as spaces. With
only 11 / 7043 rows affected (<0.2%), dropping is safer than imputing: these
correspond to brand-new customers (`tenure == 0`) where the value is genuinely
undefined, not missing at random.

**Decision:** drop the 11 rows.

In [ ]:
df = df.dropna()
df.shape

## Target distribution

In [ ]:
df["Churn"].value_counts()

In [ ]:
df["Churn"].value_counts().plot(kind="bar")
plt.title("Target Distribution")
plt.ylabel("Count")
plt.show()

~73% No / ~27% Yes -> mild imbalance. Not severe enough to warrant SMOTE or
class weights as a default.

**Decision:** no resampling. Revisit only if minority-class recall is poor.

## Categorical features

In [ ]:
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
df[cat_cols].nunique()

In [ ]:
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")

Two things to flag:

`customerID` is a unique identifier (7032 distinct values) -> no predictive value.

**Decision:** drop.

## Numeric features

In [ ]:
# SeniorCitizen is stored as 0/1 but is semantically categorical.
# Treat it as categorical for encoding consistency.
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
df[numeric_cols].describe()

In [ ]:
for col in numeric_cols:
    plt.figure()
    df[col].hist(bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

- `tenure` has lots of new and lots of long-tenured customers. 
- `MonthlyCharges` reflects service-bundle tiers. 
- `TotalCharges` is right-skewed as expected.

## Multicollinearity check

In [ ]:
df[numeric_cols].corr()

`TotalCharges` correlates 0.83 with `tenure` and 0.65 with `MonthlyCharges`. Probably because `TotalCharges ≈ tenure × MonthlyCharges`. Keeping all three would inject multicollinearity for no information gain.

**Decision:** drop `TotalCharges`.

## Churn rate by key features

The Telco Churn dataset is one of the most-studied public datasets for churn modeling, and the dominant predictors are well-documented across published analyses: `Contract`, `tenure`, and `PaymentMethod`. Rather than re-deriving this from scratch, the cells below confirm those relationships hold in our cleaned data and motivate the engineered features in the preprocessing pipeline. The goal of this notebook is preprocessing decisions, not novel feature discovery

In [ ]:
df["churn_binary"] = (df["Churn"] == "Yes").astype(int)

churn_by_contract = df.groupby("Contract")["churn_binary"].mean().sort_values(ascending=False)
churn_by_contract.plot(kind="bar")
plt.title("Churn rate by Contract type")
plt.ylabel("Churn rate")
plt.xticks(rotation=0)
plt.show()
churn_by_contract

Month-to-month contracts churn at ~43%, two-year contracts at ~3%. Contract type is the strongest single predictor in this dataset.

In [ ]:
df["tenure_bucket"] = pd.cut(
    df["tenure"], bins=[0, 6, 12, 24, 48, 72], include_lowest=True,
    labels=["0-6m", "7-12m", "13-24m", "25-48m", "49-72m"],
)
churn_by_tenure = df.groupby("tenure_bucket", observed=True)["churn_binary"].mean()
churn_by_tenure.plot(kind="bar")
plt.title("Churn rate by tenure bucket")
plt.ylabel("Churn rate")
plt.xticks(rotation=0)
plt.show()
churn_by_tenure

Strong relationship, new customers churn far more than long-tenured ones. Justifies considering tenure buckets as an engineered feature in the pipeline.

In [ ]:
churn_by_payment = df.groupby("PaymentMethod")["churn_binary"].mean().sort_values(ascending=False)
churn_by_payment.plot(kind="bar")
plt.title("Churn rate by Payment Method")
plt.ylabel("Churn rate")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()
churn_by_payment

Electronic check users churn at roughly double the rate of automatic-payment users.

## EDA Summary & Preprocessing Decisions

**Dataset:** 7032 rows × 19 features (after cleaning, excluding target)

**Target:** `Churn` (Yes/No) -> ~73% No, ~27% Yes. Mild imbalance, no resampling.

**Dropped columns:**
- `customerID` -> identifier, no predictive value
- `TotalCharges` -> multicollinear with `tenure` and `MonthlyCharges`

**Dropped rows:**
- 11 rows where `TotalCharges` was blank (all `tenure == 0`)

**Numeric columns** (`tenure`, `MonthlyCharges`):
- `StandardScaler` so that all numeric features are on the same scale, since they have very different ranges (tenure is 0–72, MonthlyCharges is ~18–119).

**Categorical columns** (`gender`, `Partner`, `Dependents`, `PhoneService`, `MultipleLines`,
`InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`,
`StreamingTV`, `StreamingMovies`, `Contract`, `PaperlessBilling`, `PaymentMethod`,
`SeniorCitizen`):
- `OneHotEncoder` so that we do not imply any ranking, as the categories are unordered. 
- `SeniorCitizen` treated as categorical despite 0/1 encoding

**Target encoding:** `Churn` mapped Yes → 1, No → 0 in the preprocessing pipeline.

**Strongest predictors observed:** `Contract`, `tenure`, `PaymentMethod`.